In [ ]:
# à faire :
#importation du fichier depuis s3 vers le service cloud
#structuration des données 
#enregistrement sur le s3 des donnees strucutrées
#detection cancer à l'aide du modele resnet18 entrainées sur le miniddsm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shantanughosh/vindr-mammogram-dataset-dicom-to-png")

print("Path to dataset files:", path)

In [ ]:
import os
endpoint_s3 = "s3/lucasvital/stat_app/"
os.system(f"mc mirror {path} {endpoint_s3}")

print(f"Fichiers synchronisés sur S3 à l'adresse : {endpoint_s3}")

In [ ]:
#il faut aussi télécharger le fichier csv des métadonnées à part
#lien du kaggle :  https://www.kaggle.com/datasets/shantanughosh/vindr-mammogram-dataset-dicom-to-png/data

In [ ]:
#on récupère le fichier avec ces commandes dans le terminal : mkdir -p ~/work/dataset_sample puis mc cp -r s3/lucasvital/stat_app/images_png/ ~/work/dataset_sample/
#on récupère aussi le fichier des metadonnées : vindr_detection_v1_folds.csv puis mc cp s3/lucasvital/stat_app/vindr_detection_v1_folds.csv ~/work/metadonnees/


In [1]:
import pandas as pd
df_metadonnees=pd.read_csv("/home/onyxia/work/metadonnees/vindr_detection_v1_folds.csv")
#print(df_metadonnees.head())
df_metadonnees.groupby("breast_birads").count()

/tmp/ipykernel_4147/4091431516.py:2: DtypeWarning: Columns (0: finding_birads) have mixed types. Specify dtype option on import or set low_memory=False.
  df_metadonnees=pd.read_csv("/home/onyxia/work/metadonnees/vindr_detection_v1_folds.csv")


,patient_id,series_id,image_id,laterality,view,height,width,breast_density,finding_categories,finding_birads,...,Focal_Asymmetry,Global_Asymmetry,Mass,Nipple_Retraction,No_Finding,Skin_Retraction,Skin_Thickening,Suspicious_Calcification,Suspicious_Lymph_Node,density
breast_birads,,,,,,,,,,,,,,,,,,,,,
BI-RADS 1,13406,13406,13406,13406,13406,13406,13406,13406,13406,0,...,13406,13406,13406,13406,13406,13406,13406,13406,13406,13406
BI-RADS 2,4676,4676,4676,4676,4676,4676,4676,4676,4676,0,...,4676,4676,4676,4676,4676,4676,4676,4676,4676,4676
BI-RADS 3,972,972,972,972,972,972,972,972,972,842,...,972,972,972,972,972,972,972,972,972,972
BI-RADS 4,1005,1005,1005,1005,1005,1005,1005,1005,1005,917,...,1005,1005,1005,1005,1005,1005,1005,1005,1005,1005
BI-RADS 5,427,427,427,427,427,427,427,427,427,370,...,427,427,427,427,427,427,427,427,427,427


In [2]:
print(df_metadonnees.columns)

Index(['patient_id', 'series_id', 'image_id', 'laterality', 'view', 'height',
       'width', 'breast_birads', 'breast_density', 'finding_categories',
       'finding_birads', 'xmin', 'ymin', 'xmax', 'ymax', 'split',
       'resized_xmin', 'resized_ymin', 'resized_xmax', 'resized_ymax', 'fold',
       'Architectural_Distortion', 'Asymmetry', 'Focal_Asymmetry',
       'Global_Asymmetry', 'Mass', 'Nipple_Retraction', 'No_Finding',
       'Skin_Retraction', 'Skin_Thickening', 'Suspicious_Calcification',
       'Suspicious_Lymph_Node', 'density'],
      dtype='str')


In [3]:
df_metadonnees.head(2)

,patient_id,series_id,image_id,laterality,view,height,width,breast_birads,breast_density,finding_categories,...,Focal_Asymmetry,Global_Asymmetry,Mass,Nipple_Retraction,No_Finding,Skin_Retraction,Skin_Thickening,Suspicious_Calcification,Suspicious_Lymph_Node,density
0,48575a27b7c992427041a82fa750d3fa,26de4993fa6b8ae50a91c8baf49b92b0,4e3a578fe535ea4f5258d3f7f4419db8.png,R,CC,3518,2800,BI-RADS 4,DENSITY C,['Mass'],...,0,0,1,0,0,0,0,0,0,2
1,48575a27b7c992427041a82fa750d3fa,26de4993fa6b8ae50a91c8baf49b92b0,dac39351b0f3a8c670b7f8dc88029364.png,R,MLO,3518,2800,BI-RADS 4,DENSITY C,['Mass'],...,0,0,1,0,0,0,0,0,0,2


In [5]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# on restructure les donées en deux dossier 0 et 1 à l'aide des étiquettes contenues dans le fichier des métadonnées
import os
import shutil
from tqdm import tqdm

os.makedirs("/home/onyxia/work/dataset_vindr",exist_ok=True)
os.makedirs("/home/onyxia/work/dataset_vindr/0-normal",exist_ok=True)
os.makedirs("/home/onyxia/work/dataset_vindr/1-cancer_benign",exist_ok=True)


def restructure_data():
    source="/home/onyxia/work/dataset_sample/"
    destination_cancer_benign="/home/onyxia/work/dataset_vindr/1-cancer_benign"
    destination_normal="/home/onyxia/work/dataset_vindr/0-normal"
    for filename in tqdm(os.listdir(source)):
        subfolder=os.path.join(source,filename)
        for image in os. listdir(subfolder):
            info_image=df_metadonnees[df_metadonnees["image_id"]==image]
            if info_image["view"].iloc[0]=="MLO":
                if info_image["breast_birads"].iloc[0]=="BI-RADS 1":
                    shutil.copy2(os.path.join(subfolder,image),destination_normal)
                else:
                    shutil.copy2(os.path.join(subfolder,image),destination_cancer_benign)
    print("restructuration terminée :)")



In [22]:
restructure_data()

100%|██████████| 5000/5000 [01:19<00:00, 63.15it/s]

restructuration terminée :)


In [ ]:
path="/home/onyxia/work/dataset_vindr"
endpoint_s3 = "s3/lucasvital/stat_app/dataset_vindr"
os.system(f"mc mirror {path} {endpoint_s3}")

print(f"Fichiers synchronisés sur S3 à l'adresse : {endpoint_s3}")

`/home/onyxia/work/dataset_vindr/0-normal/000611f8c6a44659a1813f4019241829.png` -> `s3/lucasvital/stat_app/0-normal/000611f8c6a44659a1813f4019241829.png`
`/home/onyxia/work/dataset_vindr/0-normal/000470cbf12fe2b285cba99286a9a4fa.png` -> `s3/lucasvital/stat_app/0-normal/000470cbf12fe2b285cba99286a9a4fa.png`
`/home/onyxia/work/dataset_vindr/0-normal/0027b96e3fad26237fc3b9a4e3764569.png` -> `s3/lucasvital/stat_app/0-normal/0027b96e3fad26237fc3b9a4e3764569.png`
`/home/onyxia/work/dataset_vindr/0-normal/001fb1c119340b6ae228f4ff68f9ac15.png` -> `s3/lucasvital/stat_app/0-normal/001fb1c119340b6ae228f4ff68f9ac15.png`
`/home/onyxia/work/dataset_vindr/0-normal/0033486d8e272f4914b6e6220b21297f.png` -> `s3/lucasvital/stat_app/0-normal/0033486d8e272f4914b6e6220b21297f.png`
`/home/onyxia/work/dataset_vindr/0-normal/0047aa1ae897453c44092f0552966d04.png` -> `s3/lucasvital/stat_app/0-normal/0047aa1ae897453c44092f0552966d04.png`
`/home/onyxia/work/dataset_vindr/0-normal/0054157b864fdec43a8519c33774ed68.p


== WARN: `minio.lab.sspcloud.fr` certificate will expire in 2026-02-26 12:48:48 +0000 UTC. Renew soon to avoid outage.

